**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/yang-2026"], check=True)
    os.chdir("/content/encore/entries/yang-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Yang (2026) from dandiset 000363

Paper. Yang Z (2026). Form, Timing, and Behavioral Matching of the Population Response to ALM
Photoinhibition. Research Square preprint,
[doi:10.21203/rs.3.rs-11447300/v1](https://doi.org/10.21203/rs.3.rs-11447300/v1), version 1 (posted
2026-10-22). The claims file (`claims/yang-2026.yaml`) was written from the full text of that
preprint: the main text, the Materials and Methods, the legends of Figures 1 to 3, Table 1 and the
Extended Data legends. The paper's code is in a private repository and was not read.

Dandiset. [DANDI:000363](https://dandiarchive.org/dandiset/000363/0.231012.2129), the Mesoscale
Activity Map dataset of Chen et al. (2024), version 0.231012.2129. The paper names this version
in its Methods ("Dataset" and "Code and data") and in its acknowledgments, and it is also the most
recent published version, so it is pinned. The earlier published versions (0.221127.2345 to
0.230822.0128) list the same 174 assets at sizes of 19 to 67 GB in all, against 65.7 TB here; the
pinned version adds the raw electrical series. The analysis reads only the processed arrays
(trials, events, units, video keypoints), and the raw series are never read. The versions are
listed in the first data check.

What is used. The 174 session files are surveyed for photoinhibition; the 112 late-delay
bilateral sessions are then streamed for the spike times of the "good" units, the trial events,
licks and video keypoints. Nothing is downloaded to disk. The run takes 35 to 60 minutes, depending on the runner: about
25 for streaming, and most of the rest for decoding under six seeds.

Claims. All 25 claims of the claims file are tested; none is blocked.

| Claim | Tested |
|---|---|
| session-counts | tested |
| table1-early-joint-fraction | tested |
| table1-early-axis-fractions | tested |
| axes-cosine | tested |
| fig1e-laser-vs-control-split | tested |
| fig1-1b-corrected-joint-fraction | tested |
| fig1-1c-fraction-vs-neurons | tested |
| fig1-1d-without-video-regression | tested |
| fig1b-example-session | tested |
| table1-decoding-early-vs-prelick | tested |
| fig2a-decoding-rises-after-go | tested |
| matched-reaction-time | tested |
| fig2d-choice-share-flat | tested |
| fig2e-action-share-rises | tested |
| fig2b-go-aligned-joint-fraction | tested |
| fig2-1-energy-outside-subspace | tested |
| fig3a-matched-pairs | tested |
| table1-prelick-residual | tested |
| fig3e-residual-ratio | tested |
| fig3-1ab-distances-scale | tested |
| fig3-1d-matching-keeps-distance | tested |
| fig3-1c-alm-early-residual | tested |
| fig3c-residual-time-course | tested |
| fig3c-late-residuals | tested |
| fig3b-example-plane | tested |

The shared methods M1 to M10 are those of the claims file's opening comment. The helper `mam.py`
says where each parameter comes from.

In [ ]:
import os
for _k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_k] = "1"
import json
import resource
import time
import urllib.request
import warnings

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import stats

import mam

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
T0 = time.time()
ASSETS = mam.assets()
SURVEY = mam.survey_all(ASSETS, workers=12)
print(f"surveyed {len(SURVEY)} files of dandiset {mam.DANDISET_ID} version {mam.DANDISET_VERSION} "
      f"in {time.time() - T0:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
versions = json.load(urllib.request.urlopen(
    f"https://api.dandiarchive.org/api/dandisets/{mam.DANDISET_ID}/versions/"))["results"]
print(pd.DataFrame([{"version": v["version"], "assets": v["asset_count"], "size_GB": round(v["size"] / 1e9, 1),
                     "created": v["created"][:10]} for v in versions]).to_string(index=False))
sv = pd.DataFrame([{"mouse": s["mouse"], "session": s["session"], "trials": s["trials"]["n"],
                    "units": s["n_units"], "good": s["n_good"], "photostim": s["has_photostim"],
                    "bilateral_trials": s["n_bilateral"], "median_onset": s["median_onset"],
                    "bilateral_codes": s["trials"]["bilateral_codes"],
                    "stim_trials": int(s["trials"]["stim"].sum()),
                    "stim_events": s["trials"]["photostim_events"],
                    "stim_trials_with_event": int((s["trials"]["code"] > 0).sum())} for s in SURVEY])
print("\nfiles:", len(sv), "; mice:", sv.mouse.nunique())
print("sessions with photostimulation trials:", int(sv.photostim.sum()),
      "(paper, Methods, Dataset: 168)")
print("photostimulation trials that have an onset event in the trial:",
      int(sv.stim_trials_with_event.sum()), "of", int(sv.stim_trials.sum()),
      "; events:", int(sv.stim_events.sum()))
print("sessions with a bilateral-ALM code:", int((sv.bilateral_trials > 0).sum()),
      "; codes used:", sorted({c for x in sv.bilateral_codes for c in x}))
ph = sv[sv.photostim]
print("photostimulation sessions without a code whose sites are ALM on both sides:",
      int((ph.bilateral_trials == 0).sum()))
print("\nmedian bilateral onset relative to Go (s), rounded, sessions:")
print(ph[ph.bilateral_trials > 0].median_onset.round(2).value_counts().to_string())
late = sv[(sv.median_onset - mam.LATE_ONSET).abs() <= mam.ONSET_TOL]
print("\nlate-delay bilateral sessions:", len(late), "from", late.mouse.nunique(), "mice")

168 files have photostimulation trials, as the Methods say, and every photostimulation trial has
one onset event inside it, so trials and events are on one clock and pair one to one. The
bilateral code is 6 in every session that has one (its site list holds ALM at ML -1.5 and +1.5 mm).
Median bilateral onsets fall in two groups, about -1.2 s and -0.5 s from Go, as the Methods
describe; 112 sessions from 19 mice are late-delay sessions. Seventeen photostimulation sessions
have no code whose sites are ALM on both sides and enter no analysis.

In [ ]:
# Recordings deposited twice: compare spike counts, trial counts and trial start times.
sig = {}
for s in SURVEY:
    k = (s["n_spikes"], s["trials"]["n"])
    sig.setdefault(k, []).append(s["session"])
same_counts = [v for v in sig.values() if len(v) > 1]
starts = {s["session"]: np.round(np.diff(s["trials"]["start"][:50]), 3) for s in SURVEY}
same_starts = [(a, b) for i, a in enumerate(starts) for b in list(starts)[i + 1:]
               if len(starts[a]) == len(starts[b]) and np.array_equal(starts[a], starts[b])]
print("sessions sharing spike and trial counts:", same_counts or "none")
print("sessions sharing their first 50 inter-trial intervals:", same_starts or "none")
per_day = sv.assign(day=sv.session.str[4:12]).groupby(["mouse", "day"]).size()
print("mouse-days with two sessions:", per_day[per_day > 1].to_dict())
tt_all = pd.concat([pd.DataFrame({"instr": s["trials"]["instr"], "early": s["trials"]["early"],
                                  "outcome": s["trials"]["outcome"], "auto": s["trials"]["auto_water"],
                                  "free": s["trials"]["free_water"], "go": np.isfinite(s["trials"]["go"])})
                    for s in SURVEY if s["session"] in set(late.session)])
print("\nlate-delay sessions, all trials:", len(tt_all))
print("instruction (+1 right, -1 left):", tt_all.instr.value_counts().to_dict())
print("outcome:", tt_all.outcome.value_counts().to_dict(), "; early lick:", int(tt_all.early.sum()),
      "; auto water:", int((tt_all.auto > 0).sum()), "; free water:", int((tt_all.free > 0).sum()),
      "; without a Go cue:", int((~tt_all.go).sum()))

No two files share spike and trial counts or trial timing, so no recording is deposited twice. One
mouse (sub-456772) has two sessions on each of five days; they have different trial and spike counts and
are counted as separate sessions, as in the paper's 93 ALM sessions. Trials with auto or free water
are kept, as the paper excludes none.

In [ ]:
T1 = time.time()
LATE = [s for s in SURVEY if mam.is_late_delay(s)]
S = mam.load_all(LATE, workers=4)
print(f"loaded {len(S)} sessions in {time.time() - T1:.0f} s; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

In [ ]:
rows = []
for s in S:
    tt = s["trials"]
    reg = pd.Series(s["region"]).value_counts()
    alm = np.where(s["region"] == "ALM")[0]
    bil = np.isin(tt["code"], tt["bilateral_codes"])
    ctrl = (~tt["stim"]) & (~tt["early"])

    def effect(col):
        # mean over ALM units of |laser mean - control mean| / control SD
        if len(alm) == 0:
            return np.nan, np.nan
        X = s["rates"][:, alm, col]
        sd = X[ctrl].std(0)
        ok = sd > 0
        return float(np.mean(np.abs(X[bil].mean(0) - X[ctrl].mean(0))[ok] / sd[ok])), float(np.mean((X[bil].mean(0) < X[ctrl].mean(0))[ok]))
    eff_e, down_e = effect(mam.COL_EARLY_OWN)
    eff_b, _ = effect(0)
    rows.append({"mouse": s["mouse"], "session": s["session"], **{r: int(reg.get(r, 0)) for r in mam.REGIONS},
                 "probe_labels": "; ".join(sorted(set(s["probe"]))),
                 "onset_spread_ms": 1000 * np.nanmax(np.abs(tt["onset"][bil] - s["median_onset"])),
                 "ALM_effect_early": eff_e, "ALM_effect_before": eff_b, "ALM_units_lower_early": down_e,
                 "spikes_in_trials": s["spike_range"][0] <= s["trial_range"][1] and s["spike_range"][1] >= s["trial_range"][0],
                 "unsorted_units": s["unsorted_units"],
                 "frame_ms": s["video_info"]["Jaw"]["frame_interval_ms"],
                 "jaw_det": s["video_info"]["Jaw"]["detected"], "nose_det": s["video_info"]["Nose"]["detected"],
                 "tongue_det": s["video_info"]["Tongue"]["detected"],
                 "tongue_after_lick": s["video_info"].get("tongue_after_lick", np.nan),
                 "tongue_shifted": s["video_info"].get("tongue_after_shifted", np.nan),
                 "video_covers_trials": s["video_info"]["Jaw"]["t_range"][0] <= s["trial_range"][0] + 5
                 and s["video_info"]["Jaw"]["t_range"][1] >= s["trial_range"][1] - 5})
chk = pd.DataFrame(rows)
print("good units of the five regions, per session:")
print(chk[mam.REGIONS].describe().round(1).to_string())
print("\nprobe labels of good units:", sorted({p for s in S for p in set(s["probe"])}))
print("largest deviation of a bilateral onset from the session median (ms):", round(chk.onset_spread_ms.max(), 2))
print("units with unsorted spike times:", int(chk.unsorted_units.sum()))
print("sessions whose spikes span the trials:", int(chk.spikes_in_trials.sum()), "of", len(chk))
ce = chk.dropna(subset=["ALM_effect_early"])
print("\nALM, mean |laser - control| / control SD per unit, sessions with ALM units:", len(ce))
print("  20-120 ms after onset: median", round(ce.ALM_effect_early.median(), 3), "range", ce.ALM_effect_early.round(3).agg(["min", "max"]).tolist())
print("  600 to 550 ms before Go (before onset): median", round(ce.ALM_effect_before.median(), 3), "range", ce.ALM_effect_before.round(3).agg(["min", "max"]).tolist())
print("  sessions where the early effect exceeds the effect before onset:", int((ce.ALM_effect_early > ce.ALM_effect_before).sum()), "of", len(ce))
print("  fraction of ALM units with a lower laser rate in the early window, median over sessions:", round(ce.ALM_units_lower_early.median(), 3))
print("\nvideo frame interval (ms):", chk.frame_ms.round(2).unique().tolist(),
      "; video covers the trials in", int(chk.video_covers_trials.sum()), "of", len(chk), "sessions")
print("fraction of frames detected (likelihood >= 0.9), median over sessions: jaw",
      round(chk.jaw_det.median(), 3), "nose", round(chk.nose_det.median(), 3), "tongue", round(chk.tongue_det.median(), 3))
print("tongue detected in the 50 ms after a lick:", round(chk.tongue_after_lick.median(), 3),
      "; with lick times shifted by -0.5 s:", round(chk.tongue_shifted.median(), 3),
      "; sessions where the true times give more:", int((chk.tongue_after_lick > chk.tongue_shifted).sum()), "of", len(chk))
vid = np.concatenate([s["video"] for s in S])
print("trials with no tongue detection in the choice window:", round(float(np.isnan(vid[:, 4]).mean()), 3),
      "; with no jaw detection:", round(float(np.isnan(vid[:, 0]).mean()), 4))

The region of a unit is the probe target of its electrode ("left ALM", "right Striatum" and so
on); the files also give a CCF annotation per unit (`anno_name`), not used here (claims M2,
reading a). Spike times span the trials; four units have spike times out of order (see the count above), which
the helper sorts before counting. The clock of spikes and trials is shown
by the photoinhibition itself: in every session with ALM units, the laser-control difference of
ALM units 20 to 120 ms after laser onset is larger than the difference 600 to 550 ms before Go,
before the laser (median 0.82 against 0.15 control SD), and most ALM units fire less on laser
trials in the early window. The clock of video and licks is shown by the tongue: it is detected far more often in
the 50 ms after a lick than at the same times shifted by half a second, in every session. Bilateral
onsets lie within a millisecond of the session median, so the two placements of the early window
(claims M4) give the same trials and nearly the same spikes. The tongue is rarely detected in the
choice window (no detection in 82% of trials), which bears on the video regression's handling of
missing keypoints (claims M5). In two sessions the video does not cover the span of the trials to
within 5 s; trials without keypoints take the control mean.

## Shared measures

The early-window measures for every region entry, under each combination of the open readings
that bear on them: early-window placement (M4: `own`, laser trials on their own onset; `go`, all
trials at the median onset), missing video keypoints (M5: `fill` with the control mean; `drop`
the trial from the fit), and lick direction (M3: `licks`, first lick after Go; `outcome`, from
instruction and outcome). Laser trials with an early lick are dropped (M1 reading b), the reading
under which the session counts match (claim session-counts). The first combination listed,
`own, fill, licks`, is the one reported first. Under `drop` a control trial that lacks a
detection of any keypoint leaves the choice-axis fit itself, not only the regression; as the tongue
is undetected in the choice window on 82% of trials, the choice axis then rests on about a fifth
of the control trials, which is why the `drop` readings give other fractions and cosines. A
region entry with no more such trials than the regression's seven columns is left out of the
`drop` readings (48 of 278 entries, printed below): the fit is then exact, and the residual
choice axis is round-off that differs from one machine to another.

Seeds. Every random step is run under the claim's seed, the four more seeds of the protocol
(`+1000` to `+4000`), and the paper's seed 17 (`paper`; Methods, Statistics). NumPy's generator
and the order of draws cannot be those of the paper's private code, so seed 17 does not give the
paper's splits, folds or match order; it is shown as a variant, and the outcome is the one under
the claim's seed.

In [ ]:
T2 = time.time()
READINGS = [(e, v, l) for e in ("own", "go") for v in ("fill", "drop") for l in ("licks", "outcome")]
PRIMARY = ("own", "fill", "licks")
FIVE = (0, 1000, 2000, 3000, 4000)
SEEDS = FIVE + ("paper",)
lab = lambda k: "seed 17 (paper)" if k == "paper" else f"seed +{k}"
TS = {}
for le in ("drop", "keep"):
    for l in ("licks", "outcome"):
        for s in S:
            TS[(le, l, s["session"])] = mam.trial_sets(s, le, l)
AX = {}
frac_rows = []
for (e, v, l) in READINGS:
    for s in S:
        ts = TS[("drop", l, s["session"])]
        for r in mam.REGIONS:
            ax = mam.axes(s, r, ts, early=f"early_{e}", video=v)
            if ax is None:
                continue
            f = mam.fractions(ax["d"], ax)
            if (e, v, l) == PRIMARY:
                AX[(r, s["mouse"], s["session"])] = ax
            frac_rows.append({"reading": f"{e},{v},{l}", "region": r, "mouse": s["mouse"], "session": s["session"],
                              "n_units": ax["n_units"], "joint": f["joint"], "choice": f["choice"],
                              "action": f["action"], "cos": float(ax["choice"] @ ax["action"])})
FR = pd.DataFrame(frac_rows)
# Reading b of the video regression leaves out an entry whose kept control trials are no more than the regressors.
for (e, v, l) in READINGS:
    if v != "drop":
        continue
    k = lambda rd: set(map(tuple, FR[FR.reading == rd][["region", "session"]].values))
    lost = sorted(k(f"{e},fill,{l}") - k(f"{e},drop,{l}"))
    print(f"reading {e},drop,{l}: {len(k(f'{e},drop,{l}'))} region entries; left out (too few control trials with keypoints): {len(lost)}", lost)


def mice_table(df, cols, by=("reading", "region")):
    out = []
    for keys, g in df.groupby(list(by), sort=False):
        row = dict(zip(by, keys if isinstance(keys, tuple) else (keys,)))
        for c in cols:
            m, se, nm, ns, _ = mam.mouse_summary({(a, b): x for a, b, x in zip(g.mouse, g.session, g[c])})
            row[c] = m
            row[c + "_sem"] = se
        row["mice"], row["sessions"] = nm, ns
        out.append(row)
    return pd.DataFrame(out)


print(f"{len(FR)} region entries over {len(READINGS)} readings in {time.time() - T2:.0f} s")

## session-counts

> Of the 168 sessions with photoinhibition, the late-delay bilateral sessions give ALM estimates
> from 19 mice and 93 sessions, with 13 mice and 48 sessions for striatum, 10 and 31 for thalamus,
> 16 and 71 for midbrain and 8 and 35 for medulla.

A region counts in a session when the joint fraction can be computed for it (M1, M2), under the
two M1 readings for laser trials with an early lick and the two M3 readings for lick direction.

In [ ]:
EXPECTED_COUNTS = {"ALM": (19, 93), "striatum": (13, 48), "thalamus": (10, 31), "midbrain": (16, 71), "medulla": (8, 35)}
rows = []
for le in ("drop", "keep"):
    for l in ("licks", "outcome"):
        cnt = {r: set() for r in mam.REGIONS}
        for s in S:
            ts = TS[(le, l, s["session"])]
            for r in mam.REGIONS:
                if mam.axes(s, r, ts) is not None:
                    cnt[r].add((s["mouse"], s["session"]))
        row = {"laser early-lick trials": le, "lick direction": l}
        for r in mam.REGIONS:
            row[r] = f"{len({m for m, _ in cnt[r]})} / {len(cnt[r])}"
        rows.append(row)
SC = pd.DataFrame(rows)
print("sessions with photoinhibition:", int(sv.photostim.sum()), "(expected 168 +- 16.8)")
print("mice / sessions per region:")
print(SC.to_string(index=False))
print("expected:", {r: f"{a} / {b}" for r, (a, b) in EXPECTED_COUNTS.items()})
drop_counts = SC.iloc[0]
ok = all(drop_counts[r] == f"{a} / {b}" for r, (a, b) in EXPECTED_COUNTS.items())
print("every count equal to the paper's under early-lick laser trials dropped, licks:", ok)
units_alm = FR[(FR.reading == ",".join(PRIMARY)) & (FR.region == "ALM")].n_units
print("units entering an ALM estimate per session: median", int(units_alm.median()), "range", int(units_alm.min()), "to", int(units_alm.max()))

**Result: reproduced.** 168 sessions have photoinhibition. With laser trials that had an early
lick dropped (M1 reading b), every region count equals the paper's (19 / 93, 13 / 48, 10 / 31,
16 / 71, 8 / 35) under both lick readings; keeping those trials adds one ALM, one striatum and one
midbrain session. The dropped reading is the one the other claims report first.

## table1-early-joint-fraction

> In the 20 to 120 ms after bilateral laser onset only 5 to 8% of the laser-control population
> difference lies in the joint choice-action subspace, from 0.052 in ALM (the lowest) to 0.080 in
> thalamus (the highest).

In [ ]:
EXP_JOINT = {"ALM": (0.052, 0.011), "striatum": (0.061, 0.009), "thalamus": (0.080, 0.016),
             "midbrain": (0.060, 0.007), "medulla": (0.056, 0.013)}
MT = mice_table(FR, ["joint", "choice", "action", "cos"])
jt = MT.pivot(index="reading", columns="region", values="joint")[mam.REGIONS].loc[[",".join(x) for x in READINGS]]
print("joint fraction, mean across mice:")
print(jt.round(4).to_string())
print("SEM, first reading:", MT[MT.reading == ",".join(PRIMARY)].set_index("region").joint_sem.round(4).to_dict())
for rd in jt.index:
    within = [abs(jt.loc[rd, r] - EXP_JOINT[r][0]) <= EXP_JOINT[r][1] for r in mam.REGIONS]
    print(f"{rd:20s} within tolerance: {sum(within)}/5; all below 0.10: {bool((jt.loc[rd] < 0.10).all())}; "
          f"highest {jt.loc[rd].idxmax()}, lowest {jt.loc[rd].idxmin()}")

In [ ]:
# The keypoint likelihood cut is not given by the paper; the first reading under three cuts.
rows = []
for thr in mam.LIKELIHOOD_VARIANTS:
    for s in S:
        ts = TS[("drop", "licks", s["session"])]
        for r in mam.REGIONS:
            ax = mam.axes(s, r, ts, threshold=thr)
            if ax is None:
                continue
            f = mam.fractions(ax["d"], ax)
            rows.append({"reading": f"likelihood >= {thr}", "region": r, "mouse": s["mouse"], "session": s["session"],
                         "joint": f["joint"], "choice": f["choice"], "action": f["action"], "cos": float(ax["choice"] @ ax["action"])})
THR = mice_table(pd.DataFrame(rows), ["joint", "choice", "action", "cos"])
for col in ("joint", "choice", "action", "cos"):
    print(col); print(THR.pivot(index="reading", columns="region", values=col)[mam.REGIONS].round(4).to_string())

In [ ]:
# M1 reading a: laser trials with an early lick kept, on the first readings of M3 to M5 and the "fill" readings.
rows = []
for (e, v, l) in [x for x in READINGS if x[1] == "fill"]:
    for s in S:
        ts = TS[("keep", l, s["session"])]
        for r in mam.REGIONS:
            ax = mam.axes(s, r, ts, early=f"early_{e}", video=v)
            if ax is None:
                continue
            f = mam.fractions(ax["d"], ax)
            rows.append({"reading": f"keep,{e},{v},{l}", "region": r, "mouse": s["mouse"], "session": s["session"],
                         "joint": f["joint"], "choice": f["choice"], "action": f["action"], "cos": float(ax["choice"] @ ax["action"])})
MTK = mice_table(pd.DataFrame(rows), ["joint", "choice", "action", "cos"])
for q in ("joint", "choice", "action", "cos"):
    print(q)
    print(MTK.pivot(index="reading", columns="region", values=q)[mam.REGIONS].round(4).to_string())
EXP_CH_ = {"ALM": (0.031, 0.011), "striatum": (0.044, 0.009), "thalamus": (0.059, 0.014), "midbrain": (0.030, 0.005), "medulla": (0.034, 0.011)}
EXP_AC_ = {"ALM": (0.022, 0.003), "striatum": (0.021, 0.005), "thalamus": (0.025, 0.008), "midbrain": (0.032, 0.006), "medulla": (0.026, 0.006)}
for rd, g in MTK.groupby("reading", sort=False):
    g = g.set_index("region")
    nj = sum(abs(g.loc[r, "joint"] - EXP_JOINT[r][0]) <= EXP_JOINT[r][1] for r in mam.REGIONS)
    na = sum(abs(g.loc[r, "choice"] - EXP_CH_[r][0]) <= EXP_CH_[r][1] for r in mam.REGIONS) + sum(abs(g.loc[r, "action"] - EXP_AC_[r][0]) <= EXP_AC_[r][1] for r in mam.REGIONS)
    cos_ok = abs(g.loc["thalamus", "cos"] - 0.29) <= 0.04 and abs(g.loc["ALM", "cos"] - 0.52) <= 0.03
    print(f"{rd:24s} joint within {nj}/5, below 0.10 {bool((g.joint < 0.10).all())}; axis fractions within {na}/10; "
          f"thalamus and ALM cosine within {cos_ok}; mice / sessions {g.mice.to_dict()} / {g.sessions.to_dict()}")

**Result: reproduced.** Under the first reading (own onset, filled keypoints, licks) the joint
fractions are within tolerance in all five regions and all are below 0.10; the four readings that
fill missing keypoints all match. Dropping trials without a detected keypoint from the video fit
removes most trials (the tongue is rarely seen in the delay) and gives lower fractions, within
tolerance in only one or two regions. Thalamus is the highest and ALM is not the lowest (medulla is)
under the matching readings; the criterion does not require the order. The likelihood cut for a
detected keypoint (0.5, 0.9 or 0.99) changes no fraction or cosine by more than 0.002. With
laser trials that had an early lick kept (M1 reading a, 19 / 94, 13 / 49, 10 / 31, 16 / 72, 8 / 35
entries) the joint fractions are 0.054 to 0.087, 5 of 5 within tolerance under the `licks`
reading and 4 of 5 under `outcome`, all below 0.10.

## table1-early-axis-fractions

> The choice axis alone holds 0.030 to 0.059 of the early difference and the action axis 0.021 to
> 0.032, by region as in Table 1.

In [ ]:
EXP_CH = {"ALM": (0.031, 0.011), "striatum": (0.044, 0.009), "thalamus": (0.059, 0.014), "midbrain": (0.030, 0.005), "medulla": (0.034, 0.011)}
EXP_AC = {"ALM": (0.022, 0.003), "striatum": (0.021, 0.005), "thalamus": (0.025, 0.008), "midbrain": (0.032, 0.006), "medulla": (0.026, 0.006)}
ch = MT.pivot(index="reading", columns="region", values="choice")[mam.REGIONS].loc[jt.index]
ac = MT.pivot(index="reading", columns="region", values="action")[mam.REGIONS].loc[jt.index]
print("choice fraction:"); print(ch.round(4).to_string())
print("action fraction:"); print(ac.round(4).to_string())
for rd in jt.index:
    n = sum(abs(ch.loc[rd, r] - EXP_CH[r][0]) <= EXP_CH[r][1] for r in mam.REGIONS) + \
        sum(abs(ac.loc[rd, r] - EXP_AC[r][0]) <= EXP_AC[r][1] for r in mam.REGIONS)
    print(f"{rd:20s} values within tolerance: {n}/10")

**Result: reproduced.** Under the first reading all ten choice and action fractions are within
tolerance, as under the other readings that fill missing keypoints; the readings that drop such
trials give 6 of 10. With laser trials that had an early lick kept (M1
reading a) all ten are within tolerance under the four readings that fill missing keypoints.


## axes-cosine

> The choice and action axes are not orthogonal; their cosine ranges from 0.29 in thalamus to 0.52
> in ALM.

In [ ]:
FR["abs_cos"] = FR.cos.abs()
MTc = mice_table(FR, ["cos", "abs_cos"])
for col in ("cos", "abs_cos"):
    t = MTc.pivot(index="reading", columns="region", values=col)[mam.REGIONS].loc[jt.index]
    print(f"{col} (mean across mice):"); print(t.round(3).to_string())
    for rd in t.index:
        th, al = t.loc[rd, "thalamus"], t.loc[rd, "ALM"]
        others = t.loc[rd, ["striatum", "midbrain", "medulla"]]
        between = bool(((others >= 0.29 - 0.04) & (others <= 0.52 + 0.03)).all())
        print(f"  {rd:20s} thalamus within: {abs(th - 0.29) <= 0.04}, ALM within: {abs(al - 0.52) <= 0.03}, others between: {between}")
print("SEM, first reading:", MTc[MTc.reading == ",".join(PRIMARY)].set_index("region").cos_sem.round(3).to_dict())
print("sessions with a negative cosine, first reading:", int((FR[FR.reading == ",".join(PRIMARY)].cos < 0).sum()))

**Result: reproduced.** Under the first reading the signed cosine is 0.30 in thalamus and 0.52 in
ALM, both within tolerance, and striatum, midbrain and medulla lie between them; the absolute
cosine gives the same, as almost no session has a negative cosine. The readings that drop trials
without keypoints give lower cosines (thalamus 0.24, ALM 0.38), outside tolerance; the absolute
cosine puts their thalamus (0.25) within it but not their ALM (0.39). With laser trials that had an early lick kept (M1 reading a)
the cosines are 0.295 to 0.297 in thalamus and 0.523 in ALM, also within tolerance.


## fig1e-laser-vs-control-split

> The early laser difference is larger than the difference between size-matched splits of control
> trials in every mouse, with a median ratio across mice of 152 in ALM and 3.3 to 8.7 in the other
> regions.

Twenty splits per region entry, on the first reading's early-window activity. The generator is
the claim's; the four extra seeds and seed 17 are run for the seed check. No region-entry side has
fewer than twice as many controls as laser trials (printed below), so the two small-side readings
(`disjoint`, `independent`) are the same method on different random draws, not two readings.

In [ ]:
def split_measures(seed_):
    rng = np.random.default_rng(seed_)
    out = []
    for (r, m, se), ax in AX.items():
        ts = TS[("drop", "licks", se)]
        Z, d, Q = ax["Z"]["early_own"], ax["d"], ax["Q"]
        El, Ein = float(d @ d), float(np.sum((Q.T @ d) ** 2))
        row = {"region": r, "mouse": m, "session": se, "n_units": ax["n_units"], "joint": Ein / El}
        for small in ("disjoint", "independent"):
            sp = mam.control_splits(Z, ts, rng, 20, small)
            Es = np.array([v @ v for v in sp])
            Eins = np.array([np.sum((Q.T @ v) ** 2) for v in sp])
            mv = np.mean(sp, axis=0)
            row[f"ratio_a_{small}"] = El / Es.mean()
            row[f"ratio_b_{small}"] = float(np.mean(El / Es))
            row[f"corr_a_{small}"] = (Ein - Eins.mean()) / (El - Es.mean())
            row[f"corr_b_{small}"] = (Ein - Eins.mean()) / El
            row[f"corr_c_{small}"] = Ein / El - float(np.sum((Q.T @ mv) ** 2) / (mv @ mv))
        out.append(row)
    return pd.DataFrame(out)


def mouse_means(df, col):
    return df.groupby(["region", "mouse"])[col].mean()


SPLITS = {k: split_measures(mam.seed("fig1e-laser-vs-control-split", k)) for k in SEEDS}
SP = SPLITS[0]
n_small = sum(1 for (r, m, se) in AX for sd in TS[("drop", "licks", se)]["sides"]
              if (TS[("drop", "licks", se)]["control"] & (TS[("drop", "licks", se)]["instr"] == sd)).sum()
              < 2 * (TS[("drop", "licks", se)]["laser"] & (TS[("drop", "licks", se)]["instr"] == sd)).sum())
print("region-entry sides with fewer than twice as many controls as laser trials:", n_small)
res_rows = []
for k, df in SPLITS.items():
    for ra in ("a", "b"):
        for small in ("disjoint", "independent"):
            mm = mouse_means(df, f"ratio_{ra}_{small}")
            med = mm.groupby("region").median()
            others = med.drop("ALM")
            res_rows.append({"seed+": k, "ratio": ra, "small sides": small, "ALM": med["ALM"],
                             **{r: med[r] for r in mam.REGIONS[1:]}, "lowest other": others.min(),
                             "highest other": others.max(), "min mouse ratio": mm.min(),
                             "every mouse > 1": bool((mm > 1).all())})
SPR = pd.DataFrame(res_rows)
print(SPR[SPR["seed+"] == 0].round(2).to_string(index=False))
for _, x in SPR[SPR["seed+"] == 0].iterrows():
    print(f"ratio {x['ratio']}, {x['small sides']}: ALM within {abs(x.ALM - 152) <= 15.2}, lowest within "
          f"{abs(x['lowest other'] - 3.3) <= 0.33}, highest within {abs(x['highest other'] - 8.7) <= 0.87}")
S5 = SPR[SPR["seed+"].isin(FIVE)]
ad = S5[(S5.ratio == "a") & (S5["small sides"] == "disjoint")]
print("\nover the five seeds, reading a, disjoint: ALM", ad.ALM.round(1).tolist(),
      "; lowest other", ad["lowest other"].round(2).tolist(), "; highest other", ad["highest other"].round(2).tolist())
print("lowest other over the five seeds and all readings:", round(S5["lowest other"].min(), 2), "to", round(S5["lowest other"].max(), 2))
print("\nseed 17 (paper):")
print(SPR[SPR["seed+"] == "paper"].round(2).to_string(index=False))

**Result: partially reproduced.** Every mouse in every region has a ratio above 1, the ALM
median is within tolerance (147 to 157 over the readings), the highest of the other regions
(midbrain, 7.6 to 8.8) is within it under three of the four readings (7.56 under reading a with
independent groups is below 7.83), but the lowest of the other regions (striatum or medulla) is 2.7 to
2.8 under the claim's seed and 2.6 to 2.9 over all readings and seeds, below the 2.97 the tolerance
allows. Seed 17 gives the same: ALM 145 to 158, lowest other 2.70 to 2.82, highest other 7.7 to
8.7, every mouse above 1.

## fig1-1b-corrected-joint-fraction

> After the control-split projection is removed, joint fractions remain low, with mouse means from
> about 0.003 in medulla to 0.099 in thalamus.

The splits of fig1e-laser-vs-control-split (its generator), under the three corrections (a, b, c)
and both handlings of small sides; "region" means the mean across mice, "extreme" the lowest and
highest single mouse means over all regions.

In [ ]:
rows = []
for k, df in SPLITS.items():
    for c in ("a", "b", "c"):
        for small in ("disjoint", "independent"):
            col = f"corr_{c}_{small}"
            mm = mouse_means(df, col)
            reg = mm.groupby("region").mean()
            rows.append({"seed+": k, "correction": c, "small": small, **{r: reg[r] for r in mam.REGIONS},
                         "lowest mouse": mm.min(), "lowest mouse region": mm.idxmin()[0],
                         "highest mouse": mm.max(), "highest mouse region": mm.idxmax()[0],
                         "all region means < 0.15": bool((reg < 0.15).all())})
CJ = pd.DataFrame(rows)
print(CJ[CJ["seed+"] == 0].round(4).to_string(index=False))
for _, x in CJ[CJ["seed+"] == 0].iterrows():
    reg_ok = abs(x.medulla - 0.003) <= 0.003 and abs(x.thalamus - 0.099) <= 0.0099
    ext_ok = (abs(x["lowest mouse"] - 0.003) <= 0.003 and x["lowest mouse region"] == "medulla"
              and abs(x["highest mouse"] - 0.099) <= 0.0099 and x["highest mouse region"] == "thalamus")
    print(f"correction {x.correction}, {x.small}: region means match {reg_ok}; extreme mice match {ext_ok}")
for k in SPLITS:
    c_ = CJ[CJ["seed+"] == k]
    any_ok = bool(((c_.medulla - 0.003).abs().le(0.003) & (c_.thalamus - 0.099).abs().le(0.0099)).any())
    print(f"{lab(k)}: a reading with both region means within tolerance: {any_ok}; readings with all region means < 0.15: {int(c_['all region means < 0.15'].sum())} of {len(c_)}")
C5 = CJ[CJ["seed+"].isin(FIVE)]
print("\nthalamus region mean over the five seeds and readings:", round(C5.thalamus.min(), 4), "to", round(C5.thalamus.max(), 4))
print("medulla region mean over the five seeds and readings:", round(C5.medulla.min(), 4), "to", round(C5.medulla.max(), 4))

**Result: partially reproduced.** Under the claim's seed every region mean stays below 0.15 under every correction, so
the corrected fractions remain low, but no reading gives both printed values: correction b puts
medulla near 0.003 (-0.003 to 0.002) with thalamus at 0.075 to 0.077, below the 0.089 the
tolerance allows; correction a puts thalamus near 0.11 with medulla below zero; the single mouse
extremes are far from both values. No reading matches both values under any of the five seeds. Seed 17 gives the same (no reading matches both, every region mean below 0.15 under
all six).


## fig1-1c-fraction-vs-neurons

> Joint fractions decrease with the number of neurons in a session.

In [ ]:
fp = FR[FR.reading == ",".join(PRIMARY)]
rho, p = stats.spearmanr(fp.n_units, fp.joint)
print(f"pooled over regions: Spearman rho = {rho:.3f}, p = {p:.2g}, n = {len(fp)} region entries")
for r in mam.REGIONS:
    g = fp[fp.region == r]
    rr, pp = stats.spearmanr(g.n_units, g.joint)
    print(f"  {r:9s} rho = {rr:.3f}, p = {pp:.2g}, n = {len(g)}")
for rd in jt.index:
    g = FR[FR.reading == rd]
    print(f"reading {rd:20s} pooled rho = {stats.spearmanr(g.n_units, g.joint)[0]:.3f}")

**Result: reproduced.** Pooled over regions the joint fraction falls with the number of units
(Spearman rho -0.37, p = 3e-10), and the correlation is negative in each region and under each
reading.

## fig1-1d-without-video-regression

> Without the video regression in the choice-axis fit, joint fractions remain low but change; in
> thalamus the fraction rises from 0.080 to 0.109.

In [ ]:
rows = []
for s in S:
    ts = TS[("drop", "licks", s["session"])]
    for r in mam.REGIONS:
        ax = mam.axes(s, r, ts, regress=False)
        if ax is None:
            continue
        rows.append({"region": r, "mouse": s["mouse"], "session": s["session"], "joint": mam.fractions(ax["d"], ax)["joint"]})
NV = mice_table(pd.DataFrame(rows).assign(reading="no video"), ["joint"])
WV = MT[MT.reading == ",".join(PRIMARY)].set_index("region")
NV = NV.set_index("region")
print(pd.DataFrame({"with regression": WV.joint, "without": NV.joint, "without_sem": NV.joint_sem}).loc[mam.REGIONS].round(4).to_string())
th = NV.loc["thalamus", "joint"]
print(f"thalamus without regression {th:.4f}: within 0.109 +- 0.0109: {abs(th - 0.109) <= 0.0109}; "
      f"exceeds with regression ({WV.loc['thalamus', 'joint']:.4f}): {th > WV.loc['thalamus', 'joint']}; "
      f"every region below 0.15: {bool((NV.joint < 0.15).all())}")

**Result: reproduced.** Without the video regression the thalamus joint fraction is 0.116, within
0.109 +- 0.011 and above the 0.084 with the regression; the other regions change by less than
0.004, and all stay below 0.15.

## Matched pairs and distances (shared by the claims of Figure 3)

Pairs by M9 under both match rules (`closest`, `random`), with the generator of
fig3a-matched-pairs; the four extra seeds rerun the pairs and every distance measure built on
them. Distances on z-scored activity: the pre-lick bins z-scored by control trials in each
lick-aligned bin (M9 z-score reading a; reading b, all non-early-lick control trials, is the same
set here, since only trials with a lick have a pre-lick window), units kept in all four bins.

In [ ]:
def pre_bins(s, r, ts):
    units = np.where(s["region"] == r)[0]
    Zb, K = [], np.ones(len(units), bool)
    for k in range(4):
        Z, kk = mam.zscore_window(mam.window_rates(s, units, ("pre50", k)), ts["control"])
        Zb.append(Z)
        K &= kk
    return [z[:, K] for z in Zb], int(K.sum())


PRE = {}
for (r, m, se), ax in AX.items():
    s = next(x for x in S if x["session"] == se)
    PRE[(r, m, se)] = pre_bins(s, r, TS[("drop", "licks", se)])
SBY = {s["session"]: s for s in S}


def matching_chain(k, full=True):
    out = {}
    for pick in (("closest", "random") if full else ("closest",)):
        rng = np.random.default_rng(mam.seed("fig3a-matched-pairs", k))
        pairs = {s["session"]: mam.match(TS[("drop", "licks", s["session"])], rng, pick) for s in S}
        rows = []
        for (r, m, se), ax in AX.items():
            p = pairs[se]
            if not p:
                continue
            Zb, nk = PRE[(r, m, se)]
            row = {"region": r, "mouse": m, "session": se, "n_pre_units": nk, "n_units": ax["n_units"]}
            for miss in ("drop", "keep"):
                for comb in ("mean", "concat"):
                    res, lc, cc = mam.residual(Zb, p, miss, comb)
                    row[f"res_{miss}_{comb}"], row[f"lc_{miss}_{comb}"], row[f"cc_{miss}_{comb}"] = res, lc, cc
            row["early_res"], row["early_lc"], row["early_cc"] = mam.residual([ax["Z"]["early_own"]], p)
            ts = TS[("drop", "licks", se)]
            units = np.where(SBY[se]["region"] == r)[0]
            e50 = []
            K = np.ones(len(units), bool)
            for b in range(2):
                Z, kk = mam.zscore_window(mam.window_rates(SBY[se], units, ("early50", b)), ts["control"])
                e50.append(Z)
                K &= kk
            row["early50_res"] = mam.residual([z[:, K] for z in e50], p)[0]
            if full:
                Zc, sel = mam.curve_units(SBY[se], units, ts["control"])
                row["curve"] = np.array([mam.residual([Zc[b]], p)[0] for b in range(mam.N_GO)])
                row["curve_units"] = int(sel.sum())
            rows.append(row)
        out[pick] = (pairs, pd.DataFrame(rows))
    return out


T3 = time.time()
MATCH = {0: matching_chain(0)}
for k in SEEDS[1:]:
    MATCH[k] = matching_chain(k)
PAIRS, MD = MATCH[0]["closest"]
print(f"matching and distances under five seeds and seed 17 in {time.time() - T3:.0f} s")

## fig3a-matched-pairs

> Matching laser and control trials within session and instructed side on lick direction and a
> reaction-time difference of at most 100 ms retains 2,676 pairs, with a median absolute
> reaction-time difference of 1.3 ms and a largest of 99 ms.

In [ ]:
def pair_stats(pairs, md):
    allp = [(se, p) for se, ps in pairs.items() for p in ps]
    rt = lambda se, i: SBY[se]["trials"]["first_lick"][i]
    d = np.array([abs(rt(se, p[0]) - rt(se, p[1])) for se, p in allp]) * 1000
    by_region = sum(len(pairs[se]) for se in md.session)
    return {"pairs (sessions once)": len(allp), "pairs (summed over region entries)": by_region,
            "sessions with pairs": sum(1 for v in pairs.values() if v), "median |dRT| ms": np.median(d),
            "max |dRT| ms": d.max(), "pairs without a second control": sum(p[2] < 0 for _, p in allp),
            "dropped sessions (<10 pairs)": sum(1 for v in pairs.values() if not v)}


rows = []
for k in MATCH:
    for pick in MATCH[k]:
        rows.append({"seed+": k, "pick": pick, **pair_stats(*MATCH[k][pick])})
PS = pd.DataFrame(rows)
print(PS.round(2).to_string(index=False))
x = PS[(PS["seed+"] == 0) & (PS.pick == "closest")].iloc[0]
print(f"\nclosest, sessions once: count within 2676 +- 267.6: {abs(x['pairs (sessions once)'] - 2676) <= 267.6}; "
      f"median within 1.3 +- 0.13: {abs(x['median |dRT| ms'] - 1.3) <= 0.13}; largest within 99 +- 9.9: {abs(x['max |dRT| ms'] - 99) <= 9.9}")
print("matched mice / sessions per region (Table 1: 19 / 87, 12 / 42, 10 / 29, 15 / 64, 8 / 32):",
      {r: f"{g.mouse.nunique()} / {len(g)}" for r, g in MD.groupby("region", sort=False)})

**Result: reproduced.** With the closest match, 2,667 pairs are retained in 98 sessions (each
session counted once), with a median reaction-time difference of 1.3 ms and a largest of 99.1 ms,
all within tolerance and under all five seeds and seed 17 (2,664 to 2,667 pairs, largest 95.4
to 99.1 ms); a random match within the caliper gives a median
near 43 ms, so the closest match is the paper's. Matched counts per region are 19 / 87, 12 / 41,
10 / 29, 15 / 63 and 8 / 32, against the paper's 19 / 87, 12 / 42, 10 / 29, 15 / 64 and 8 / 32.

## matched-reaction-time

> On the matched laser trials the median reaction time is 0.29 s, with quartiles 0.22 and 0.39 s,
> so the pre-lick window falls mostly after Go.

In [ ]:
rows = []
for k in MATCH:
    pairs, md_ = MATCH[k]["closest"]
    a = np.array([SBY[se]["trials"]["first_lick"][p[0]] for se, ps in pairs.items() for p in ps])
    b = np.array([SBY[se]["trials"]["first_lick"][p[0]] for se in md_.session for p in pairs[se]])
    for name, v in (("a, once per session", a), ("b, once per region entry", b)):
        q = np.percentile(v, [25, 50, 75])
        rows.append({"seed+": k, "reading": name, "n": len(v), "q25": q[0], "median": q[1], "q75": q[2],
                     "pre-lick window after Go (fraction of its 200 ms)": float(np.mean(np.clip(v, 0, 0.2) / 0.2))})
MR = pd.DataFrame(rows)
print(MR.round(4).to_string(index=False))
x = MR.iloc[0]
print(f"within tolerance: median {abs(x['median'] - 0.29) <= 0.029}, q25 {abs(x.q25 - 0.22) <= 0.022}, q75 {abs(x.q75 - 0.39) <= 0.039}")

**Result: reproduced.** The matched laser trials have a median reaction time of 0.29 s with
quartiles 0.22 and 0.38 s, within tolerance under both readings, all five seeds and seed 17; on average 94% of
the 200 ms pre-lick window lies after Go.

## table1-prelick-residual

> Every mouse has a positive pre-lick residual (laser-control minus control-control distance), with
> means of 1.52 in ALM, 1.38 in striatum, 2.99 in thalamus, 0.76 in midbrain and 0.47 in medulla.

In [ ]:
EXP_RES = {"ALM": (1.52, 0.14), "striatum": (1.38, 0.51), "thalamus": (2.99, 0.84), "midbrain": (0.76, 0.10), "medulla": (0.47, 0.13)}
rows = []
for k in MATCH:
    for pick, (pairs, md_) in MATCH[k].items():
        for miss in ("drop", "keep"):
            for comb in ("mean", "concat"):
                col = f"res_{miss}_{comb}"
                mm = md_.groupby(["region", "mouse"])[col].mean()
                reg = mm.groupby("region").agg(["mean", "sem"])
                row = {"seed+": k, "pick": pick, "second missing": miss, "bins": comb}
                for r in mam.REGIONS:
                    row[r] = reg.loc[r, "mean"]
                row["within"] = sum(abs(reg.loc[r, "mean"] - EXP_RES[r][0]) <= EXP_RES[r][1] for r in mam.REGIONS)
                row["mice not positive"] = int((mm <= 0).sum())
                rows.append(row)
RES = pd.DataFrame(rows)
print(RES[RES["seed+"] == 0].round(3).to_string(index=False))
print("\nSEM across mice, closest, drop, mean:",
      MD.groupby(["region", "mouse"]).res_drop_mean.mean().groupby("region").sem().round(3).to_dict())
p_ = RES[(RES.pick == "closest") & (RES["second missing"] == "drop") & (RES.bins == "mean")]
p0 = p_[p_["seed+"].isin(FIVE)]
print("closest, drop, mean, over five seeds:", {r: (round(p0[r].min(), 3), round(p0[r].max(), 3)) for r in mam.REGIONS},
      "; within", p0.within.tolist(), "; mice not positive", p0["mice not positive"].tolist())
print("closest, drop, mean, seed 17 (paper):", p_[p_["seed+"] == "paper"][mam.REGIONS + ["within", "mice not positive"]].round(3).to_dict("records"))

**Result: reproduced.** With the closest match, pairs without a second control dropped and per-bin
distances averaged, every mouse has a positive residual and all five means are within tolerance
(ALM 1.55, striatum 1.36, thalamus 2.91, midbrain 0.76, medulla 0.53), under all five seeds and seed 17 (ALM 1.525, medulla 0.516). The
concatenated-bin reading roughly doubles the residuals and matches in no region.

## fig3e-residual-ratio

> Divided by the control-control distance, the pre-lick residual is 0.111 in ALM, 0.093 in
> striatum, 0.142 in thalamus, 0.063 in midbrain and 0.036 in medulla.

In [ ]:
EXP_RAT = {"ALM": (0.111, 0.009), "striatum": (0.093, 0.031), "thalamus": (0.142, 0.034), "midbrain": (0.063, 0.007), "medulla": (0.036, 0.010)}
rows = []
for k in MATCH:
    for pick, (pairs, md_) in MATCH[k].items():
        for miss in ("drop", "keep"):
            for comb in ("mean", "concat"):
                res, cc = f"res_{miss}_{comb}", f"cc_{miss}_{comb}"
                g = md_.groupby(["region", "mouse"])
                ra = (g[res].mean() / g[cc].mean()).groupby("region").mean()
                rb = (md_[res] / md_[cc]).groupby([md_.region, md_.mouse]).mean().groupby("region").mean()
                for name, v in (("a, mouse means first", ra), ("b, session ratios", rb)):
                    rows.append({"seed+": k, "pick": pick, "second missing": miss, "bins": comb, "ratio": name,
                                 **{r: v[r] for r in mam.REGIONS},
                                 "within": sum(abs(v[r] - EXP_RAT[r][0]) <= EXP_RAT[r][1] for r in mam.REGIONS)})
RAT = pd.DataFrame(rows)
print(RAT[RAT["seed+"] == 0].round(4).to_string(index=False))
p_ = RAT[(RAT.pick == "closest") & (RAT["second missing"] == "drop") & (RAT.bins == "mean")]
print("\nclosest, drop, mean over five seeds, within:", p_[p_["seed+"].isin(FIVE)].within.tolist(),
      "; seed 17 (paper):", p_[p_["seed+"] == "paper"].within.tolist())
print("SEM, reading a:", (MD.groupby(["region", "mouse"]).res_drop_mean.mean() / MD.groupby(["region", "mouse"]).cc_drop_mean.mean()).groupby("region").sem().round(4).to_dict())

**Result: reproduced.** Under the first readings both ratio readings put all five values within
tolerance (ALM 0.113, striatum 0.092, thalamus 0.136, midbrain 0.064, medulla 0.042), under all five seeds and seed 17.

## fig3-1ab-distances-scale

> Raw distance scales with the number of neurons, and the control-control distance is already
> larger in thalamus than in midbrain.

In [ ]:
cc_m = MD.groupby(["region", "mouse"]).cc_drop_mean.mean().groupby("region").agg(["mean", "sem"])
lc_m = MD.groupby(["region", "mouse"]).lc_drop_mean.mean().groupby("region").agg(["mean", "sem"])
print("pre-lick distances, mean across mice:")
print(pd.DataFrame({"control-control": cc_m["mean"], "cc_sem": cc_m["sem"], "laser-control": lc_m["mean"]}).loc[mam.REGIONS].round(2).to_string())
rho, p = stats.spearmanr(MD.n_pre_units, MD.cc_drop_mean)
print(f"Spearman, control-control distance against units, pooled: rho = {rho:.3f}, p = {p:.2g}, n = {len(MD)}")
rho2, p2 = stats.spearmanr(MD.n_pre_units, MD.lc_drop_mean)
print(f"Spearman, laser-control distance against units, pooled: rho = {rho2:.3f}, p = {p2:.2g}")
print("thalamus control-control above midbrain:", cc_m.loc["thalamus", "mean"] > cc_m.loc["midbrain", "mean"])
print("units per pre-lick entry, mean by region:", MD.groupby("region").n_pre_units.mean().round(0).to_dict())

**Result: reproduced.** The control-control distance rises with the number of units (Spearman rho
0.99, p < 1e-200) and is larger in thalamus (20.0) than in midbrain (12.1), mouse means.

## fig3-1d-matching-keeps-distance

> Matching lick direction and reaction time does not remove the pre-lick laser-control distance.

Unmatched pairs: each matched laser trial with a random control trial of the same session and
side that has a lick, drawn without replacement and without regard to lick direction or reaction
time, with the claim's generator.

In [ ]:
def unmatched(seed_, md, pairs):
    rng = np.random.default_rng(seed_)
    rows = []
    for _, x in md.iterrows():
        se, r = x.session, x.region
        ts = TS[("drop", "licks", se)]
        Zb, _ = PRE[(r, x.mouse, se)]
        X = np.stack(Zb, 2)
        lasers = [p[0] for p in pairs[se]]
        up = []
        for sd in ts["sides"]:
            Ls = [l for l in lasers if ts["instr"][l] == sd]
            C = np.where(ts["control"] & (ts["instr"] == sd) & np.isfinite(ts["rt"]))[0]
            ch = rng.choice(C, min(len(Ls), len(C)), replace=False)
            up += list(zip(Ls, ch))
        up = np.array(up)
        rows.append(float(np.sqrt(((X[up[:, 0]] - X[up[:, 1]]) ** 2).sum(1)).mean(1).mean()))
    return pd.Series(rows, index=md.index)


UNM = {k: unmatched(mam.seed("fig3-1d-matching-keeps-distance", k), MATCH[k]["closest"][1], MATCH[k]["closest"][0]) for k in SEEDS}
MD["unmatched"] = UNM[0]
g = MD.groupby(["region", "mouse"])[["lc_drop_mean", "cc_drop_mean", "unmatched"]].mean().groupby("region").mean().loc[mam.REGIONS]
g["matched lc - cc"] = g.lc_drop_mean - g.cc_drop_mean
g["unmatched - matched lc"] = g.unmatched - g.lc_drop_mean
print(g.round(3).to_string())
print("matched laser-control above matched control-control in every region:", bool((g["matched lc - cc"] > 0).all()))
def unm_change(k):
    md_ = MATCH[k]["closest"][1].assign(unmatched=UNM[k])
    gg = md_.groupby(["region", "mouse"])[["lc_drop_mean", "cc_drop_mean", "unmatched"]].mean().groupby("region").mean()
    return gg.unmatched - gg.lc_drop_mean, gg.lc_drop_mean - gg.cc_drop_mean


ch = {k: unm_change(k) for k in SEEDS}
print("unmatched minus matched laser-control, over five seeds:",
      {r: (round(min(ch[k][0][r] for k in FIVE), 3), round(max(ch[k][0][r] for k in FIVE), 3)) for r in mam.REGIONS})
print("matched laser-control above control-control in every region, per seed:", {lab(k): bool((ch[k][1] > 0).all()) for k in SEEDS})
print("seed 17 (paper): unmatched minus matched", ch["paper"][0].round(3).to_dict(), "; matched lc - cc", ch["paper"][1].round(3).to_dict())

**Result: reproduced.** After matching, the laser-control distance stays above the
control-control distance in every region, by the residuals of table1-prelick-residual. Matching lowers
the laser-control distance only a little: the unmatched distance is 0.3 to 1.0 above the matched
one (0.23 to 0.96 over the five seeds, 0.39 to 0.83 under seed 17). The matched laser-control
distance exceeds the control-control distance in every region under all six seeds.

## fig3-1c-alm-early-residual

> In ALM most of the separation comes earlier; the ALM residual on the same pairs is 12.6 in the 20
> to 120 ms window against 1.52 before the lick.

The early window is scored two ways, as the paper does not say whether it is divided into bins:
as one 100 ms window (M4), and divided into two 50 ms bins with the per-bin distances averaged, as
for the pre-lick window (Methods, Matched distance: "The primary window was the 200 ms before the
lick, divided into 50 ms bins. The same pairs were scored in the early window"; Methods, Time
courses: "Distance used 50 ms bins").

In [ ]:
rows = []
for k in MATCH:
    md_ = MATCH[k]["closest"][1]
    for col in ("early_res", "early50_res", "res_drop_mean"):
        mm = md_.groupby(["region", "mouse"])[col].mean()
        rows.append({"seed+": k, "measure": col, **mm.groupby("region").mean().to_dict(), "ALM_sem": mm["ALM"].sem()})
ER = pd.DataFrame(rows)
print(ER[ER["seed+"] == 0].round(3).to_string(index=False))
e = ER[(ER["seed+"] == 0) & (ER.measure == "early_res")].iloc[0]
pre = ER[(ER["seed+"] == 0) & (ER.measure == "res_drop_mean")].iloc[0]
e50 = ER[(ER["seed+"] == 0) & (ER.measure == "early50_res")].iloc[0]
print(f"ALM early residual {e.ALM:.2f}: within 12.6 +- 2.0: {abs(e.ALM - 12.6) <= 2.0}; ratio to pre-lick {e.ALM / pre.ALM:.1f} (>= 4: {e.ALM / pre.ALM >= 4})")
print(f"two 50 ms bins: {e50.ALM:.2f}, within: {abs(e50.ALM - 12.6) <= 2.0}")
E5 = ER[ER["seed+"].isin(FIVE)]
print("ALM early residual over five seeds: one window", E5[E5.measure == "early_res"].ALM.round(2).tolist(),
      "; two 50 ms bins", E5[E5.measure == "early50_res"].ALM.round(2).tolist())
print("seed 17 (paper): one window", ER[(ER["seed+"] == "paper") & (ER.measure == "early_res")].ALM.round(2).tolist(),
      "; two 50 ms bins", ER[(ER["seed+"] == "paper") & (ER.measure == "early50_res")].ALM.round(2).tolist())
print(f"two 50 ms bins, ratio to pre-lick {e50.ALM / pre.ALM:.1f}")

**Result: reproduced, resting on an assumption.** Scored in two 50 ms bins, as every other
distance of the paper is, the ALM residual in the early window is 12.59 (SEM 2.0), within 12.6 +-
2.0, and 8.1 times the pre-lick residual of 1.55, above the factor of 4. Scored as one 100 ms
window it is 18.4 (SEM 2.7), outside tolerance (18.3 to 18.4 over the five seeds, 18.31 under seed 17), and the outcome
would be partially reproduced. The binned reading is adopted, as the paper computes all its
distances in 50 ms bins (12.56 under seed 17). The claims file does not name this binning as
open; this is asked of a reader (questions/yang-2026.yaml).

## fig3c-residual-time-course

> On the Go-aligned residual curve ALM peaks in the bin that begins at laser onset (21.3) and then
> declines; thalamus is negative in the two bins after onset (-1.2 and -1.5) and peaks at 11.2 in
> the bin centered 25 ms before Go; midbrain peaks at 2.1 shortly after onset, striatum at 2.8 in
> the bin centered 25 ms after Go, and medulla at 1.3.

In [ ]:
CENT = np.round((mam.GO_EDGES[:-1] + 0.025) * 1000).astype(int)


def curve_means(md_):
    out = {}
    for r in mam.REGIONS:
        g = md_[md_.region == r]
        mm = np.array([np.mean(np.stack(gg.curve.values), 0) for _, gg in g.groupby("mouse")])
        out[r] = (mm.mean(0), mm.std(0, ddof=1) / np.sqrt(len(mm)), mm)
    return out


CM = {k: curve_means(MATCH[k]["closest"][1]) for k in MATCH}
c0 = CM[0]
print("bin centres (ms from Go):", CENT.tolist())
for r in mam.REGIONS:
    print(f"{r:9s}", " ".join(f"{v:5.2f}" for v in c0[r][0]))
print("units per entry on the curve, median:", int(MD.curve_units.median()), "; entries with fewer than 20 units passing every bin:",
      int((MD.curve_units < 20).sum()))


def curve_checks(c):
    pk = {r: int(np.argmax(c[r][0])) for r in mam.REGIONS}
    vals = {"ALM peak": (c["ALM"][0][pk["ALM"]], 21.3, 2.9), "thal bin1": (c["thalamus"][0][2], -1.2, 0.3),
            "thal bin2": (c["thalamus"][0][3], -1.5, 0.7), "thal peak": (c["thalamus"][0][pk["thalamus"]], 11.2, 2.1),
            "mid peak": (c["midbrain"][0][pk["midbrain"]], 2.1, 0.4), "str peak": (c["striatum"][0][pk["striatum"]], 2.8, 0.9),
            "med peak": (c["medulla"][0][pk["medulla"]], 1.3, 0.5)}
    bins_ok = {"ALM at onset (-475)": CENT[pk["ALM"]] == -475, "thalamus at -25": CENT[pk["thalamus"]] == -25,
               "striatum at +25": CENT[pk["striatum"]] == 25, "midbrain within 200 ms after onset": -500 < CENT[pk["midbrain"]] < -300,
               "thalamus bins 1-2 negative": bool(c["thalamus"][0][2] < 0 and c["thalamus"][0][3] < 0)}
    within = {k: abs(v - e) <= t for k, (v, e, t) in vals.items()}
    return pk, vals, bins_ok, within


pk, vals, bins_ok, within = curve_checks(c0)
print("\npeak bins (centre, ms):", {r: int(CENT[i]) for r, i in pk.items()}, "; medulla peak at", int(CENT[pk["medulla"]]))
for k, (v, e, t) in vals.items():
    print(f"  {k:10s} {v:6.2f}  expected {e} +- {t}  within {within[k]}")
print("bin checks:", bins_ok, "; values within:", sum(within.values()), "of 7")
print("ALM after the onset bin:", np.round(c0["ALM"][0][2:8], 2).tolist(), "; SEM at onset", round(c0["ALM"][1][2], 2),
      "; thalamus SEM bins 1-2", np.round(c0["thalamus"][1][2:4], 2).tolist())
for k in SEEDS[1:]:
    pk_, vals_, bins_, within_ = curve_checks(CM[k])
    print(f"{lab(k)}: bins all hold {all(bins_.values())}, values within {sum(within_.values())} of 7")

**Result: reproduced.** The ALM residual peaks in the bin that begins at laser onset (21.3) and then
declines; thalamus is negative in the two bins after onset (-1.0 and -1.4) and peaks at 11.3 in the
bin centred 25 ms before Go; midbrain peaks at 2.1 in the bin centred 425 ms before Go, striatum at
2.9 in the bin centred 25 ms after Go and medulla at 1.3. All seven values are within tolerance and
all named bins hold, under all five seeds and seed 17.

## fig3c-late-residuals

> From 0.23 to 0.38 s after Go the residuals are 0.8 to 1.4 in ALM, 0.9 to 2.1 in thalamus, 0.3 to
> 0.8 in striatum and below 0.5 in midbrain and medulla.

In [ ]:
late_idx = [i for i, c in enumerate(CENT) if 225 <= c <= 375]
BOUNDS = {"ALM": (0.72, 1.54), "thalamus": (0.81, 2.31), "striatum": (0.27, 0.88), "midbrain": (-np.inf, 0.55), "medulla": (-np.inf, 0.55)}
print("bins centred", [int(CENT[i]) for i in late_idx], "ms")
for k in CM:
    ok = {}
    for r in mam.REGIONS:
        v = CM[k][r][0][late_idx]
        ok[r] = bool(((v >= BOUNDS[r][0]) & (v <= BOUNDS[r][1])).all())
        if k == 0:
            print(f"  {r:9s} {np.round(v, 2).tolist()}  bounds {BOUNDS[r]}  inside {ok[r]}")
    out_ = {r: np.round(CM[k][r][0][late_idx], 2).tolist() for r in mam.REGIONS if not ok[r]}
    print(f"{lab(k)}: regions inside {sum(ok.values())} of 5; outside: {out_ or 'none'}")

**Result: reproduced.** Under the claim's seed every region lies inside its widened range in the
four bins centred 225 to 375 ms after Go (ALM 0.79 to 1.33, thalamus 0.82 to 1.95, striatum 0.29 to
0.80, midbrain and medulla at most 0.47). Under seeds +1000, +3000 and +4000 and under seed 17 the
thalamus value in the bin centred 375 ms (0.76 to 0.81) falls just below the widened lower bound of
0.81, so four of five regions hold and the outcome there would be partially reproduced; under +2000
all five hold. The outcome is seed-sensitive.


## Decoding (shared by table1-decoding-early-vs-prelick and fig2a-decoding-rises-after-go)

Lick direction is decoded under both readings of M8. Reading (b), laser and control trials
together, is reported first, as nearly every region entry passes the Methods' rule of 40 trials
and 15 per lick direction with it. Reading (a), laser trials alone ("There is no paired trial
without laser", Results), is also run on the region entries that pass the rule with laser trials
alone; the cell prints how many do, and the bilateral and laser trials per session. Under reading
(a) a laser trial's feature is its deviation from the mean of all control trials of its side;
no control trial is decoded, so this mean is not estimated inside the fold. The early and pre-lick
windows choose the penalty inside each training fold; the Go-aligned curve uses C = 1. Units are
those kept in the axes windows, the pre-lick window and every 100 ms Go bin. Z-scoring (M4) uses
the control trials of each window before cross-validation; it uses no labels. The claim seeds
are used for the folds, and four more seeds and seed 17 for the seed check.

In [ ]:
T4 = time.time()
G100 = list(range(len(mam.go100_starts())))
DEC_IN = []
laser_only_ok = 0
for s in S:
    ts = TS[("drop", "licks", s["session"])]
    for r in mam.REGIONS:
        if (r, s["mouse"], s["session"]) not in AX:
            continue
        ax = mam.axes(s, r, ts, extra_windows=["pre"] + [("go100", k) for k in G100])
        if ax is None:
            continue
        ld = ts["lick_dir"]
        m_laser = ts["laser"] & (ld != 0) & np.isin(ts["instr"], ts["sides"])
        ok_laser = bool(m_laser.sum() >= 40 and (ld[m_laser] == 1).sum() >= 15 and (ld[m_laser] == -1).sum() >= 15)
        laser_only_ok += ok_laser
        m = (ts["laser"] | ts["control"]) & (ld != 0) & np.isin(ts["instr"], ts["sides"])
        ok = m.sum() >= 40 and (ld[m] == 1).sum() >= 15 and (ld[m] == -1).sum() >= 15
        lk = np.isfinite(ts["rt"])
        d_pre = mam._side_diff(ax["Z"]["pre"], ts["laser"] & lk, ts["control"] & lk, ts["instr"], ts["sides"], ts["laser"] & lk)
        d_go = [mam._side_diff(ax["Z"][("go100", k)], ts["laser"], ts["control"], ts["instr"], ts["sides"], ts["laser"]) for k in G100]
        DEC_IN.append({"region": r, "mouse": s["mouse"], "session": s["session"], "ok": ok, "ax": ax, "m": m,
                       "y": ld[m], "instr": ts["instr"][m], "ctrl": ts["control"][m], "n_trials": int(m.sum()),
                       "ok_laser": ok_laser, "m_laser": m_laser, "control": ts["control"], "all_instr": ts["instr"],
                       "early_frac": mam.fractions(ax["d"], ax), "pre_frac": mam.fractions(d_pre, ax),
                       "go_frac": [mam.fractions(d, ax) for d in d_go], "n_units": ax["n_units"]})
print("region entries:", len(DEC_IN), "; passing the decoding rule with laser and control trials:", sum(x["ok"] for x in DEC_IN),
      "; with laser trials alone:", laser_only_ok)
print("passing with laser trials alone, mice / entries per region:",
      {r: f"{len({x['mouse'] for x in DEC_IN if x['ok_laser'] and x['region'] == r})} / {sum(x['ok_laser'] and x['region'] == r for x in DEC_IN)}" for r in mam.REGIONS})
nb_ = pd.Series([int(np.isin(s["trials"]["code"], s["trials"]["bilateral_codes"]).sum()) for s in S])
nl_ = pd.Series([int((TS[("drop", "licks", s["session"])]["laser"] & (s["trials"]["lick_dir"] != 0)).sum()) for s in S])
print(f"bilateral trials per late-delay session: median {nb_.median():.0f}, range {nb_.min()} to {nb_.max()}; "
      f"laser trials without an early lick and with a lick after Go: median {nl_.median():.0f}, range {nl_.min()} to {nl_.max()}; "
      f"sessions with at least 40 such trials: {int((nl_ >= 40).sum())} of {len(nl_)}")


def run_decoding(k, curve=True):
    jobs, meta = [], []
    for i, x in enumerate(DEC_IN):
        if not x["ok"]:
            continue
        Z = x["ax"]["Z"]
        for w in ("early_own", "pre"):
            jobs.append(delayed(mam.decode)(Z[w][x["m"]], x["y"], x["instr"], x["ctrl"], mam.seed("table1-decoding-early-vs-prelick", k)))
            meta.append((i, w))
        if curve:
            for b in G100:
                jobs.append(delayed(mam.decode)(Z[("go100", b)][x["m"]], x["y"], x["instr"], x["ctrl"],
                                                mam.seed("fig2a-decoding-rises-after-go", k), fixed_C=1.0))
                meta.append((i, b))
    out = Parallel(n_jobs=4)(jobs)
    acc = {}
    for (i, w), v in zip(meta, out):
        acc.setdefault(w, {})[i] = v
    return acc


def run_decoding_laser(k):
    """M8 reading a: laser trials alone, each as its deviation from the mean of its side's control trials."""
    jobs, meta = [], []
    for i, x in enumerate(DEC_IN):
        if not x["ok_laser"]:
            continue
        ml, ctrl, ins = x["m_laser"], x["control"], x["all_instr"]
        il = ins[ml]
        y = TS[("drop", "licks", x["session"])]["lick_dir"][ml]
        for w in ["early_own", "pre"] + [("go100", b) for b in G100]:
            Z = x["ax"]["Z"][w]
            X = Z[ml].copy()
            for sd in (-1, 1):
                if (il == sd).any():
                    X[il == sd] -= np.nanmean(Z[ctrl & (ins == sd)], axis=0)
            if w in ("early_own", "pre"):
                jobs.append(delayed(mam.decode)(X, y, il, np.zeros(len(y), bool), mam.seed("table1-decoding-early-vs-prelick", k), precentered=True))
            else:
                jobs.append(delayed(mam.decode)(X, y, il, np.zeros(len(y), bool), mam.seed("fig2a-decoding-rises-after-go", k),
                                                fixed_C=1.0, precentered=True))
            meta.append((i, w if w in ("early_own", "pre") else w[1]))
    out = Parallel(n_jobs=4)(jobs)
    acc = {}
    for (i, w), v in zip(meta, out):
        acc.setdefault(w, {})[i] = v
    return acc


DEC = {0: run_decoding(0)}
print(f"decoding, claim seeds, in {time.time() - T4:.0f} s")
for k in SEEDS[1:]:
    DEC[k] = run_decoding(k)
print(f"decoding, five seeds and seed 17, in {time.time() - T4:.0f} s")
DECL = {k: run_decoding_laser(k) for k in SEEDS}
print(f"decoding of laser trials alone (reading a), six seeds, in {time.time() - T4:.0f} s")


def dec_summary(acc, w):
    vals = {(DEC_IN[i]["mouse"], DEC_IN[i]["session"], DEC_IN[i]["region"]): v for i, v in acc[w].items()}
    out = {}
    for r in mam.REGIONS:
        out[r] = mam.mouse_summary({(m, se): v for (m, se, rr), v in vals.items() if rr == r})
    return out

## table1-decoding-early-vs-prelick

> Balanced accuracy for lick direction rises from 0.567 to 0.611 in the early window to 0.788 to
> 0.839 in the 200 ms before the first lick.

In [ ]:
EXP_EA = {"ALM": (0.584, 0.006), "striatum": (0.602, 0.014), "thalamus": (0.580, 0.011), "midbrain": (0.567, 0.008), "medulla": (0.611, 0.019)}
EXP_PA = {"ALM": (0.788, 0.025), "striatum": (0.802, 0.034), "thalamus": (0.797, 0.045), "midbrain": (0.796, 0.025), "medulla": (0.839, 0.027)}
for k in DEC:
    e, p = dec_summary(DEC[k], "early_own"), dec_summary(DEC[k], "pre")
    n_in = sum(abs(e[r][0] - EXP_EA[r][0]) <= EXP_EA[r][1] for r in mam.REGIONS) + sum(abs(p[r][0] - EXP_PA[r][0]) <= EXP_PA[r][1] for r in mam.REGIONS)
    rise = all(p[r][0] > e[r][0] for r in mam.REGIONS)
    if k == 0:
        print(pd.DataFrame({"early": {r: e[r][0] for r in mam.REGIONS}, "early_sem": {r: e[r][1] for r in mam.REGIONS},
                            "pre-lick": {r: p[r][0] for r in mam.REGIONS}, "pre_sem": {r: p[r][1] for r in mam.REGIONS},
                            "mice / sessions": {r: f"{e[r][2]} / {e[r][3]}" for r in mam.REGIONS},
                            "early within": {r: abs(e[r][0] - EXP_EA[r][0]) <= EXP_EA[r][1] for r in mam.REGIONS},
                            "pre within": {r: abs(p[r][0] - EXP_PA[r][0]) <= EXP_PA[r][1] for r in mam.REGIONS}}).round(4).to_string())
    print(f"{lab(k)}: values within tolerance {n_in}/10; rise in every region {rise}; early",
          {r: round(e[r][0], 3) for r in mam.REGIONS})
print("\nreading a, laser trials alone:")
for k in DECL:
    e, p = dec_summary(DECL[k], "early_own"), dec_summary(DECL[k], "pre")
    n_in = sum(bool(abs(e[r][0] - EXP_EA[r][0]) <= EXP_EA[r][1]) for r in mam.REGIONS) + sum(bool(abs(p[r][0] - EXP_PA[r][0]) <= EXP_PA[r][1]) for r in mam.REGIONS)
    rise = all(p[r][0] > e[r][0] for r in mam.REGIONS)
    if k == 0:
        print(pd.DataFrame({"early": {r: e[r][0] for r in mam.REGIONS}, "early_sem": {r: e[r][1] for r in mam.REGIONS},
                            "pre-lick": {r: p[r][0] for r in mam.REGIONS}, "pre_sem": {r: p[r][1] for r in mam.REGIONS},
                            "mice / sessions": {r: f"{e[r][2]} / {e[r][3]}" for r in mam.REGIONS}}).round(4).to_string())
    print(f"{lab(k)}: values within tolerance {n_in}/10; rise in every region (a region without entries fails) {rise}")

**Result: partially reproduced.** Decoding laser and control trials together (M8 reading b),
accuracy rises from the early window to the 200 ms before the lick in every region, and all five
pre-lick values are within tolerance (0.794 to 0.856), but the early values (0.561 to 0.584) are
lower than the paper's, and those of ALM (0.574), striatum (0.584) and medulla (0.581) are outside
tolerance: 7 of 10 values within, where 8 are needed. The other seeds give 6 or 7 of 10 and seed
17 gives 6. Decoding laser trials alone (M8 reading a), on the 26 entries that pass the Methods'
rule (ALM 7 mice / 10 entries, thalamus 1 / 2), the rise holds in every region, but only 0 of 10
values are within tolerance under the claim's seed (1 to 4 over the other seeds and seed 17), as
the pre-lick accuracies are lower (0.67 to 0.77). Neither reading gives eight values within
tolerance.


## fig2a-decoding-rises-after-go

> On the Go-aligned grid balanced accuracy stays near 0.55 to 0.58 through the delay and rises
> after Go, to 0.73 to 0.77 in the 100 ms bin centered 250 ms after Go.

The 100 ms bins step by 50 ms; the non-overlapping reading is the subset of bins starting at -600,
-500, ... ms. The delay bins are those within the 600 ms before Go (starting -600 to -100 ms).

In [ ]:
starts = np.round(mam.go100_starts() * 1000).astype(int)
k250 = int(np.where(starts == 200)[0][0])
delay = [i for i, st in enumerate(starts) if st + 100 <= 0]
delay_nonover = [i for i in delay if (starts[i] + 600) % 100 == 0]
for rd_name, DD in (("reading b, laser and control", DEC), ("reading a, laser alone", DECL)):
    print(rd_name)
    for k in DD:
        curves = {r: np.array([dec_summary(DD[k], b)[r][0] for b in G100]) for r in mam.REGIONS}
        if k == 0:
            print("bin starts (ms):", starts.tolist())
            for r in mam.REGIONS:
                print(f"{r:9s}", " ".join(f"{v:.3f}" for v in curves[r]))
        v250 = {r: curves[r][k250] for r in mam.REGIONS}
        for name, dl in (("overlapping", delay), ("non-overlapping", delay_nonover)):
            dvals = np.concatenate([curves[r][dl] for r in mam.REGIONS])
            ok250 = all(0.657 <= v <= 0.847 for v in v250.values())
            okd = bool(((dvals >= 0.495) & (dvals <= 0.638)).all())
            rise = all(curves[r][k250] > curves[r][dl].max() for r in mam.REGIONS)
            print(f"{lab(k)}, {name}: 250 ms bin {[round(v, 3) for v in v250.values()]} in range {ok250}; delay bins "
                  f"{dvals.min():.3f} to {dvals.max():.3f} in range {okd}; 250 ms bin above every delay bin {rise}")

**Result: reproduced.** Decoding laser and control trials together (M8 reading b), in the bin
centred 250 ms after Go the region means are 0.72 to 0.78, inside 0.657 to 0.847; every delay bin
lies between 0.54 and 0.59, inside 0.495 to 0.638; and the 250 ms bin exceeds every delay bin in
every region, under both bin readings, all five seeds and seed 17. Decoding laser trials alone
(M8 reading a, 26 entries) gives noisy curves (delay bins 0.41 to 0.69, the 250 ms bin 0.63 in
thalamus and 0.635 in ALM, both below 0.657) that fail every part of the criterion; the outcome rests on reading (b).


## fig2d-choice-share-flat

> The choice-axis share of the laser-minus-control vector does not rise with decoding accuracy;
> early-window means are 0.035 to 0.046, and before the lick the share falls or stays flat in four
> regions and rises only in medulla, from 0.036 to 0.058, with pre-lick means of 0.025 (ALM), 0.019
> (striatum), 0.012 (thalamus), 0.034 (midbrain) and 0.058 (medulla).

The pre-lick vector on trials with a lick, z-scored in the 200 ms window; the axes of M5 on the
units kept in every window of the decoding entries. Session sets: (a) the entries that pass the
decoding rule, (b) the entries with matched pairs.

In [ ]:
matched_keys = set(zip(MD.region, MD.mouse, MD.session))
SETS = {"a, decoding": [x for x in DEC_IN if x["ok"]],
        "a, decoding laser trials alone": [x for x in DEC_IN if x["ok_laser"]],
        "b, matched": [x for x in DEC_IN if (x["region"], x["mouse"], x["session"]) in matched_keys]}


def share_table(entries, q):
    out = {}
    for r in mam.REGIONS:
        e = mam.mouse_summary({(x["mouse"], x["session"]): x["early_frac"][q] for x in entries if x["region"] == r})
        p = mam.mouse_summary({(x["mouse"], x["session"]): x["pre_frac"][q] for x in entries if x["region"] == r})
        out[r] = {"early": e[0], "early_sem": e[1], "pre": p[0], "pre_sem": p[1], "mice": e[2], "sessions": e[3],
                  "rises (> pre SEM)": p[0] - e[0] > p[1], "mouse-level change": p[0] - e[0]}
    return pd.DataFrame(out).T


EXP_PC = {"ALM": (0.025, 0.004), "striatum": (0.019, 0.005), "thalamus": (0.012, 0.002), "midbrain": (0.034, 0.008), "medulla": (0.058, 0.014)}
for name, ent in SETS.items():
    t = share_table(ent, "choice")
    print(f"session set {name}:")
    print(t.to_string())
    early_ok = bool(((t.early >= 0.0315) & (t.early <= 0.0506)).all())
    pre_ok = {r: abs(t.loc[r, "pre"] - EXP_PC[r][0]) <= EXP_PC[r][1] for r in mam.REGIONS}
    med_ok = abs(t.loc["medulla", "early"] - 0.036) <= 0.0036
    rises = t["rises (> pre SEM)"]
    print(f"  early means in 0.0315-0.0506: {early_ok}; pre-lick within: {pre_ok}; medulla early within 0.036: {med_ok}; "
          f"rises only in medulla: {bool(rises['medulla'] and not rises.drop('medulla').any())}; "
          f"regions not rising (SEM rule): {int((~rises.astype(bool)).sum())}; "
          f"regions whose mean does not rise at all: {int((t['mouse-level change'].astype(float) <= 0).sum())}")

**Result: partially reproduced, resting on an assumption.** In sets (a) and (b) the choice share
before the lick is lower than in the early window in ALM, striatum and thalamus, rises slightly in
midbrain (by 0.006, less than its SEM of 0.006 to 0.007) and rises in medulla by more than its SEM,
so it "rises only in medulla" by the criterion's SEM rule; the pre-lick means of ALM, thalamus,
midbrain and medulla are within tolerance. Striatum's pre-lick mean (0.025) is outside 0.019 +-
0.005, and the early means (0.031 to 0.053 in set a, 0.033 to 0.068 in set b) fall outside 0.0315
to 0.0506. The partial clause ("does not rise in at least four regions") is read with the same SEM
rule, an assumption: counting any increase of the mean as a rise, three regions do not rise in
every set and the outcome would be not reproduced. This is asked of a reader
(questions/yang-2026.yaml). On the entries that pass the decoding rule with laser trials alone
(M8 reading a, 26 entries) the early means are 0.006 to 0.147 and no pre-lick mean is within
tolerance.


## fig2e-action-share-rises

> The action-axis share rises from 0.016 to 0.030 in the early window to 0.040 to 0.122 before the
> lick, largest in medulla (0.122).

In [ ]:
for name, ent in SETS.items():
    t = share_table(ent, "action")
    print(f"session set {name}:")
    print(t[["early", "early_sem", "pre", "pre_sem", "mice", "sessions"]].to_string())
    rise = bool((t.pre > t.early).all())
    largest = t.pre.astype(float).idxmax()
    vals = {"medulla pre": (t.loc["medulla", "pre"], 0.122, 0.015), "lowest pre": (t.pre.min(), 0.040, 0.004),
            "lowest early": (t.early.min(), 0.016, 0.0016), "highest early": (t.early.max(), 0.030, 0.003)}
    print(f"  rise in every region: {rise}; largest pre-lick: {largest}; within:",
          {k: (round(v, 4), abs(v - e) <= tol) for k, (v, e, tol) in vals.items()})

**Result: partially reproduced.** In sets (a) and (b) the action share rises from the early
window to the pre-lick window in every region, it is largest in medulla (0.125 to 0.128, within
tolerance), and the highest early value is within tolerance; the lowest pre-lick value (thalamus,
0.030) and the lowest early value (0.019 to 0.022) are outside tolerance. On the entries that pass
the decoding rule with laser trials alone (26 entries) the share does not rise in striatum or
thalamus and no value is within tolerance.


## fig2b-go-aligned-joint-fraction

> On the Go-aligned grid the joint fraction stays about 0.04 to 0.08 through the delay; in the bin
> centered 250 ms after Go it is 0.165 in medulla, 0.102 in midbrain, 0.072 in ALM, 0.075 in
> striatum and 0.056 in thalamus.

Session sets: the early-fraction entries (all region entries of the decoding table, which are those
of Table 1), and the distance-curve entries (those with matched pairs).

In [ ]:
EXP_GJ = {"medulla": (0.165, 0.028), "midbrain": (0.102, 0.016), "ALM": (0.072, 0.010), "striatum": (0.075, 0.015), "thalamus": (0.056, 0.012)}
GSETS = {"early-fraction entries": DEC_IN, "distance-curve entries": SETS["b, matched"]}
for name, ent in GSETS.items():
    curves, sems = {}, {}
    for r in mam.REGIONS:
        per = [mam.mouse_summary({(x["mouse"], x["session"]): x["go_frac"][b]["joint"] for x in ent if x["region"] == r}) for b in G100]
        curves[r] = np.array([p[0] for p in per])
        sems[r] = np.array([p[1] for p in per])
    print(f"{name}: bin starts {starts.tolist()}")
    for r in mam.REGIONS:
        print(f"  {r:9s}", " ".join(f"{v:.3f}" for v in curves[r]))
    v250 = {r: (round(curves[r][k250], 4), round(sems[r][k250], 4), abs(curves[r][k250] - EXP_GJ[r][0]) <= EXP_GJ[r][1]) for r in mam.REGIONS}
    print("  250 ms bin (mean, SEM, within):", v250)
    for dname, dl in (("overlapping", delay), ("non-overlapping", delay_nonover)):
        dv = np.concatenate([curves[r][dl] for r in mam.REGIONS])
        out = {r: [round(float(v), 3) for v in curves[r][dl] if not 0.036 <= v <= 0.088] for r in mam.REGIONS}
        print(f"  delay bins, {dname}: {dv.min():.3f} to {dv.max():.3f}; all in 0.036-0.088: {bool(((dv >= 0.036) & (dv <= 0.088)).all())}; outside: {out}")

**Result: partially reproduced.** On the early-fraction entries all five values in the bin
centred 250 ms after Go are within tolerance (medulla 0.184, midbrain 0.107, striatum 0.083, ALM
0.079, thalamus 0.050), but the delay bins are not all inside 0.036 to 0.088: medulla is 0.099
in the first bin, thalamus 0.034 to 0.035 in several bins, and ALM 0.035 in one bin with 50 ms
steps. On the distance-curve entries ALM
(0.085) and medulla (0.199) are outside tolerance in the 250 ms bin.

## fig2-1-energy-outside-subspace

> Most of the population difference stays outside the coding subspace; in ALM the energy per neuron
> outside the subspace rises from 0.09 before laser onset to 26 in the 100 ms bin that begins at
> onset and then falls, while the other regions stay near or below 1.

Reading (a): units and z-scoring of the distance curve (M10), 100 ms bins as the mean of two
z-scored 50 ms bins, axes restricted to those units. Reading (b): as for the Go-aligned joint
fraction, each 100 ms bin z-scored by its control trials. Both on the early-fraction entries.

In [ ]:
k_on = int(np.where(starts == -500)[0][0])
k_pre = int(np.where(starts == -600)[0][0])
EN = {"a": [], "b": []}
for x in DEC_IN:
    ax = x["ax"]
    for gf in [x["go_frac"]]:
        EN["b"].append({"region": x["region"], "mouse": x["mouse"], "session": x["session"],
                        "out": np.array([f["energy"] * (1 - f["joint"]) for f in gf]) / x["n_units"],
                        "in": np.array([f["energy"] * f["joint"] for f in gf]) / x["n_units"]})
    s = SBY[x["session"]]
    ts = TS[("drop", "licks", x["session"])]
    units = np.where(s["region"] == x["region"])[0]
    Zc, sel = mam.curve_units(s, units, ts["control"])
    a0 = AX[(x["region"], x["mouse"], x["session"])]
    inax = np.isin(units[sel], a0["units"])
    pos = [int(np.where(a0["units"] == u)[0][0]) for u in units[sel][inax]]
    c, a = a0["choice"][pos], a0["action"][pos]
    Q, _ = np.linalg.qr(np.c_[c / np.linalg.norm(c), a / np.linalg.norm(a)])
    eo, ei = [], []
    for b in G100:
        Z = (Zc[b][:, inax] + Zc[b + 1][:, inax]) / 2
        d = mam._side_diff(Z, ts["laser"], ts["control"], ts["instr"], ts["sides"], ts["laser"])
        e_in = float(np.sum((Q.T @ d) ** 2))
        eo.append((d @ d - e_in) / inax.sum())
        ei.append(e_in / inax.sum())
    EN["a"].append({"region": x["region"], "mouse": x["mouse"], "session": x["session"], "out": np.array(eo), "in": np.array(ei)})
for rd in ("a", "b"):
    df = pd.DataFrame(EN[rd])
    print(f"reading {rd}:")
    res = {}
    for r in mam.REGIONS:
        g = df[df.region == r]
        mo = np.array([np.mean(np.stack(gg.out.values), 0) for _, gg in g.groupby("mouse")])
        mi = np.array([np.mean(np.stack(gg["in"].values), 0) for _, gg in g.groupby("mouse")])
        res[r] = (mo.mean(0), mo.std(0, ddof=1) / np.sqrt(len(mo)), mi.mean(0))
        print(f"  {r:9s} out:", " ".join(f"{v:.3f}" for v in res[r][0]))
        print(f"  {'':9s} in: ", " ".join(f"{v:.3f}" for v in res[r][2]))
    alm = res["ALM"]
    others_max = {r: round(float(res[r][0].max()), 3) for r in mam.REGIONS[1:]}
    print(f"  ALM before onset {alm[0][k_pre]:.3f} (0.09 +- 0.009: {abs(alm[0][k_pre] - 0.09) <= 0.009}); onset bin {alm[0][k_on]:.2f} +- {alm[1][k_on]:.2f} "
          f"(26 +- 10: {abs(alm[0][k_on] - 26) <= 10}); falls after onset: {bool(alm[0][k_on + 1] < alm[0][k_on] and alm[0][-1] < alm[0][k_on])}")
    print(f"  other regions, largest bin: {others_max}; all below 1.1: {all(v < 1.1 for v in others_max.values())}")
    print(f"  outside above inside at onset in every region: {all(res[r][0][k_on] > res[r][2][k_on] for r in mam.REGIONS)}")
    nonover = [i for i, st in enumerate(starts) if (st + 600) % 100 == 0]
    print(f"  other regions, largest non-overlapping bin: { {r: round(float(res[r][0][nonover].max()), 3) for r in mam.REGIONS[1:]} }")

**Result: partially reproduced.** Under reading (b) the ALM energy per neuron outside the
subspace is 25.8 +- 10.2 in the bin that begins at laser onset, within 26 +- 10, and then falls;
outside exceeds inside in every region at onset; with non-overlapping bins the other regions stay
below 1.1 (largest 0.84, striatum), while with 50 ms steps thalamus reaches 1.19 in one bin. The
ALM value before onset is 0.053, outside 0.09 +- 0.009. Reading (a) gives an ALM peak of 9.8,
outside tolerance.

## fig1b-example-session

> The pre-specified ALM example session is ses-20200922T135318, with 130 neurons and a joint fraction
> of 0.053, chosen as the session whose joint fraction is closest to the ALM mouse mean.

Candidates: ALM entries of the early-fraction analysis that also enter the distance curve (have
matched pairs), with at least the median neuron count of that set. "Neuron count" is read as the
units that enter the joint fraction; the count of good ALM units is shown as well.

In [ ]:
alm_mean = MT[(MT.reading == ",".join(PRIMARY)) & (MT.region == "ALM")].joint.iloc[0]
cand = []
for (r, m, se), ax in AX.items():
    if r != "ALM" or (r, m, se) not in matched_keys:
        continue
    cand.append({"session": se, "mouse": m, "n_units": ax["n_units"], "good_ALM_units": ax["n_region_units"],
                 "joint": mam.fractions(ax["d"], ax)["joint"]})
cand = pd.DataFrame(cand)
for count in ("n_units", "good_ALM_units"):
    med = cand[count].median()
    c = cand[cand[count] >= med].copy()
    c["distance"] = (c.joint - alm_mean).abs()
    c = c.sort_values(["distance", count, "session"], ascending=[True, False, True])
    print(f"neuron count = {count}: ALM mouse mean {alm_mean:.5f}; median {med}; {len(c)} candidates; closest three:")
    print(c.head(3).round(5).to_string(index=False))
ex = cand[cand.session == "ses-20200922T135318"].iloc[0]
print(f"\nses-20200922T135318: {ex.n_units} units in the fraction ({ex.good_ALM_units} good ALM units), joint fraction {ex.joint:.4f}; "
      f"count within 130 +- 13: {abs(ex.n_units - 130) <= 13}; fraction within 0.053 +- 0.0053: {abs(ex.joint - 0.053) <= 0.0053}")

print("\nthe rule under every reading of M3, M4 and M5 (candidates: ALM entries with matched pairs, at least the median unit count of the reading):")
rows = []
for rd in [",".join(x) for x in READINGS]:
    mean_ = MT[(MT.reading == rd) & (MT.region == "ALM")].joint.iloc[0]
    c = FR[(FR.reading == rd) & (FR.region == "ALM")].copy()
    c = c[[("ALM", m, se) in matched_keys for m, se in zip(c.mouse, c.session)]]
    c = c[c.n_units >= c.n_units.median()]
    c["distance"] = (c.joint - mean_).abs()
    c = c.sort_values(["distance", "n_units", "session"], ascending=[True, False, True]).reset_index(drop=True)
    ex_ = c[c.session == "ses-20200922T135318"]
    rows.append({"reading": rd, "ALM mouse mean": mean_, "candidates": len(c), "selected": c.session.iloc[0],
                 "selected distance": c.distance.iloc[0], "paper's session rank": int(ex_.index[0]) + 1 if len(ex_) else None,
                 "paper's session distance": float(ex_.distance.iloc[0]) if len(ex_) else np.nan,
                 "paper's session units": int(ex_.n_units.iloc[0]) if len(ex_) else None,
                 "paper's session joint": float(ex_.joint.iloc[0]) if len(ex_) else np.nan})
print(pd.DataFrame(rows).round(5).to_string(index=False))

**Result: partially reproduced.** ses-20200922T135318 has 130 units and a joint fraction of 0.0528,
both within tolerance, but the rule selects ses-20210419T161140 (130 units, 0.0529), which is closer
to the ALM mouse mean (0.0549) by 0.0001; the paper's session is second. Under the other seven
readings of M3 to M5 the rule never selects the paper's session: the readings that fill missing
keypoints select ses-20210419T161140 and rank the paper's session second or third, and the
readings that drop trials without keypoints select ses-20201230T112558 and rank the paper's
session 34th of 39 (its joint fraction is 0.001 there).


## fig3b-example-plane

> In the example session matched laser and control trajectories separate along the early
> off-subspace direction and come closer again at Go; the displayed plane holds 25% of the
> session's early matched difference, the first principal component of control activity orthogonal
> to the choice axis 4%.

In ses-20200922T135318 (the paper's session) and, as a variant, in the session the rule of
fig1b-example-session selects. "At Go" is the 100 ms bin centred on Go.

In [ ]:
k_go = int(np.where(starts == -50)[0][0])


def plane(se, direction, pcs):
    s = SBY[se]
    ts = TS[("drop", "licks", se)]
    ax = mam.axes(s, "ALM", ts, extra_windows=[("go100", k_go)])
    p = np.array(PAIRS[se])
    Ze, Zg = ax["Z"]["early_own"], ax["Z"][("go100", k_go)]
    c, Q = ax["choice"], ax["Q"]
    dm = Ze[p[:, 0]].mean(0) - Ze[p[:, 1]].mean(0)
    base = ax["d"] if direction == "unmatched" else dm
    u = base - Q @ (Q.T @ base)
    u /= np.linalg.norm(u)
    Pm = np.c_[c, u]
    frac = float(np.sum((Pm.T @ dm) ** 2) / (dm @ dm))
    ctrl = np.where(ts["control"])[0] if pcs == "all controls" else p[:, 1]
    X = Ze[ctrl] - Ze[ctrl].mean(0)
    X = X - np.outer(X @ c, c)
    pc1 = np.linalg.svd(X, full_matrices=False)[2][0]
    frac_pc = float((pc1 @ dm) ** 2 / (dm @ dm))
    dg = Zg[p[:, 0]].mean(0) - Zg[p[:, 1]].mean(0)
    sep_e, sep_g = float(np.linalg.norm(Pm.T @ dm)), float(np.linalg.norm(Pm.T @ dg))
    return {"session": se, "direction": direction, "PC trials": pcs, "pairs": len(p), "units": ax["n_units"],
            "plane fraction": frac, "PC1 fraction": frac_pc, "separation early": sep_e, "separation at Go": sep_g}


rows = [plane(se, d_, pc) for se in ("ses-20200922T135318", "ses-20210419T161140")
        for d_ in ("unmatched", "matched") for pc in ("all controls", "matched controls")]
PL = pd.DataFrame(rows)
print(PL.round(4).to_string(index=False))
for _, x in PL.iterrows():
    print(f"{x.session} {x.direction:9s} {x['PC trials']:16s}: plane within 0.25 +- 0.025 {abs(x['plane fraction'] - 0.25) <= 0.025}; "
          f"PC1 within 0.04 +- 0.004 {abs(x['PC1 fraction'] - 0.04) <= 0.004}; closer at Go {x['separation at Go'] < x['separation early']}")

**Result: partially reproduced.** In ses-20200922T135318 the matched laser and control means are
closer at Go than in the early window in the displayed plane (separation 0.87 against 4.11 with the
unmatched direction), as in the session the rule selects. The plane holds 70% (unmatched direction)
or 97% (matched direction) of the early matched difference, against the paper's 25%, and the first
principal component holds 3.5% (all controls) or 8.5% (matched controls), both outside 0.04 +-
0.004.

## Summary

| Claim | Outcome |
|---|---|
| session-counts | reproduced |
| table1-early-joint-fraction | reproduced |
| table1-early-axis-fractions | reproduced |
| axes-cosine | reproduced |
| fig1e-laser-vs-control-split | partially reproduced |
| fig1-1b-corrected-joint-fraction | partially reproduced |
| fig1-1c-fraction-vs-neurons | reproduced |
| fig1-1d-without-video-regression | reproduced |
| fig1b-example-session | partially reproduced |
| table1-decoding-early-vs-prelick | partially reproduced |
| fig2a-decoding-rises-after-go | reproduced |
| matched-reaction-time | reproduced |
| fig2d-choice-share-flat | partially reproduced (resting on an assumption) |
| fig2e-action-share-rises | partially reproduced |
| fig2b-go-aligned-joint-fraction | partially reproduced |
| fig2-1-energy-outside-subspace | partially reproduced |
| fig3a-matched-pairs | reproduced |
| table1-prelick-residual | reproduced |
| fig3e-residual-ratio | reproduced |
| fig3-1ab-distances-scale | reproduced |
| fig3-1d-matching-keeps-distance | reproduced |
| fig3-1c-alm-early-residual | reproduced (resting on an assumption) |
| fig3c-residual-time-course | reproduced |
| fig3c-late-residuals | reproduced (seed-sensitive) |
| fig3b-example-plane | partially reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper's code is private, so every open detail
was taken from the claims file's readings or the paper's text: laser trials with an early lick
are dropped (the reading under which the session counts match exactly); regions are the probe
targets in the files; a video keypoint counts as detected at likelihood 0.9 or more, and trials
without a detection take the control mean (dropping them removes most trials, as the tongue is
rarely seen in the delay); the closest control within the caliper is matched (a random match gives
a median reaction-time difference of 43 ms, not the paper's 1.3 ms); the penalty is chosen by an
inner 5-fold search; the early-window residual is scored in two 50 ms bins (an assumption). Random draws use the registry's claim seeds; the paper's seed 17 is run as a
variant, but NumPy's generator and the order of draws cannot be the paper's, so splits, folds and
match order differ from the paper's even where the method is the same. Seed 17 gives the same
outcome as the claim's seed for every claim but fig3c-late-residuals (four of five regions). Lick
direction is decoded from laser and control trials together (M8 reading b); decoding laser trials
alone (reading a) is possible on only 26 region entries and matches less well.

What was found about the dataset. The pinned version holds what the paper describes: 168
sessions with photoinhibition, 112 late-delay bilateral sessions from 19 mice, and per-region
counts of mice and sessions equal to Table 1. No recording is deposited twice. The trial table,
the photostimulation events, the spikes, the licks and the video share one clock, by the contrasts
in the data check. Four units have spike times out of order. The region of a unit is given only as
the probe's target (`brain_regions` of the electrode location) and as a CCF annotation; neither is
described as the region label of the paper. The video keypoints have a likelihood per frame but no
detection flag.

In [ ]:
print(f"total run time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")